## Exposure Fairness — Signal Computation & Re-ranking

In [35]:
import numpy as np
import pandas as pd


def minmax_normalize(series):
    s = series.astype(float)
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or lo == hi:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)


def build_item_signal_table(df, item_col="title", save_col="save", listen_ratio_col="listen_ratio"):
    df = df.copy()
    df["save_binary"] = (df[save_col] == "yes").astype(int)
    df[listen_ratio_col] = pd.to_numeric(df[listen_ratio_col], errors="coerce")
    return (
        df.groupby(item_col, as_index=False)
          .agg(
              plays=(item_col, "size"),
              avg_listen_ratio=(listen_ratio_col, "mean"),
              save_rate=("save_binary", "mean"),
          )
    )


def compute_exposure_fairness_signals(item_stats, item_col="title", alpha=0.7, beta = 0.8):
    """alpha = weight for listen ratio; (1 - alpha) = weight for save rate."""
    """beta = weight for quality; (1 - alpha) = bonus for underexposed."""
    df = item_stats.copy()
    df["exposure_score"] = minmax_normalize(df["plays"])
    df["listen_time_score"] = minmax_normalize(df["avg_listen_ratio"])
    df["save_score"] = minmax_normalize(df["save_rate"])
    df["quality_score"] = alpha * df["listen_time_score"] + (1 - alpha) * df["save_score"]
    df["inclusion_score"] = beta * df["quality_score"] + (1 - beta) * (1 - df["exposure_score"])
    return df[[item_col, "plays", "avg_listen_ratio", "save_rate",
               "exposure_score", "listen_time_score", "save_score",
               "quality_score", "inclusion_score"]]



## Load Data & merge calculated inclusion score to podcast database

In [36]:
synthetic_view_history = pd.read_csv("data/synthetic_view_history_all_users.csv")

item_stats = build_item_signal_table(synthetic_view_history)
item_signal_table = compute_exposure_fairness_signals(item_stats)

item_signal_table.sort_values(by="inclusion_score", ascending=False).head()

bbc_seen = (pd.read_csv("data/bbc_seen_items.csv").merge(item_signal_table[["title", "inclusion_score"]], on="title", how="left"))

bbc_seen.to_csv("data/bbc_seen_items.csv", index=False)
